# Problem 2: MNIST Class Implementation and Testing
This NumPy implementation loads the supplied pretrained weights (784 → 50 → 100 → 10). It uses sigmoid hidden layers and softmax output. It reads the supplied IDX data and tests your own handwritten digits with the required messages and `imshow`.

**Still required from you:** write each digit 0–9 five times and save the 50 images as `mnist/digits/0_1.png` through `mnist/digits/9_5.png`. No supplied MNIST image is represented as your handwriting. This notebook reports missing files until you provide them.

In [1]:
from pathlib import Path
import struct
import pickle
import re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
BASE = Path.cwd() / 'mnist'

In [2]:
class Mnist:
    def __init__(self, weight_path):
        # Load only the trusted weight file supplied with the assignment.
        with open(weight_path, 'rb') as stream:
            self.network = pickle.load(stream)
        shapes = {'W1': (784, 50), 'b1': (50,), 'W2': (50, 100),
                  'b2': (100,), 'W3': (100, 10), 'b3': (10,)}
        for key, shape in shapes.items():
            if np.shape(self.network[key]) != shape:
                raise ValueError(f'Invalid shape for {key}: expected {shape}')

    @staticmethod
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -709, 709)))

    @staticmethod
    def softmax(x):
        e = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e / np.sum(e, axis=-1, keepdims=True)

    def predict(self, x):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim not in (1, 2) or x.shape[-1] != 784:
            raise ValueError('Pass one flattened 784-pixel image or a batch (N, 784).')
        if not np.isfinite(x).all() or np.any(x < 0) or np.any(x > 1):
            raise ValueError('Input values must be normalized to [0, 1].')
        w = self.network
        z1 = self.sigmoid(x @ w['W1'] + w['b1'])
        z2 = self.sigmoid(z1 @ w['W2'] + w['b2'])
        return self.softmax(z2 @ w['W3'] + w['b3'])

    @staticmethod
    def load_idx(image_path, label_path):
        with open(image_path, 'rb') as stream:
            magic, count, rows, cols = struct.unpack('>IIII', stream.read(16))
            if magic != 2051 or (rows, cols) != (28, 28):
                raise ValueError('Invalid MNIST image IDX file.')
            pixels = np.frombuffer(stream.read(), dtype=np.uint8)
        if pixels.size != count * rows * cols:
            raise ValueError('Image file length does not match its header.')
        with open(label_path, 'rb') as stream:
            magic, label_count = struct.unpack('>II', stream.read(8))
            labels = np.frombuffer(stream.read(), dtype=np.uint8)
        if magic != 2049 or label_count != count or len(labels) != count:
            raise ValueError('Invalid or mismatched label IDX file.')
        return pixels.reshape(count, 784).astype(np.float32) / 255.0, labels

    def accuracy(self, x, y, batch_size=100):
        correct = 0
        for start in range(0, len(x), batch_size):
            predicted = np.argmax(self.predict(x[start:start+batch_size]), axis=1)
            correct += np.count_nonzero(predicted == y[start:start+batch_size])
        return correct / len(x)

    @staticmethod
    def load_handwritten(image_path):
        with Image.open(image_path) as image:
            if image.mode != 'L' or image.size != (28, 28):
                raise ValueError(f'{image_path}: expected a 28 x 28 grayscale (L) PNG.')
            pixels = np.asarray(image, dtype=np.float32)
        # Match MNIST: bright digit on a dark background.
        border = np.concatenate([pixels[0], pixels[-1], pixels[:, 0], pixels[:, -1]])
        if np.median(border) > 127:
            pixels = 255.0 - pixels
        return pixels.astype(np.uint8), pixels.reshape(784) / 255.0

    def test_handwritten(self, folder):
        folder = Path(folder)
        expected = [folder / f'{digit}_{version}.png'
                    for digit in range(10) for version in range(1, 6)]
        missing = [p.name for p in expected if not p.exists()]
        if missing:
            print(f'Handwritten test incomplete: {len(missing)} of 50 images missing.')
            print('Missing:', ', '.join(missing))
        results = []
        for path in expected:
            if not path.exists():
                continue
            actual = int(path.stem.split('_')[0])
            pixels, x = self.load_handwritten(path)
            predicted = int(np.argmax(self.predict(x)))
            plt.figure(figsize=(2, 2))
            plt.imshow(pixels, cmap='gray', vmin=0, vmax=255)
            plt.title(path.name)
            plt.axis('off')
            plt.show()
            if predicted == actual:
                print(f'Success: Image {path.name} is for digit {actual} is recognized as {predicted}.')
            else:
                print(f'Fail: Image {path.name} is for digit {actual} but the inference result is {predicted}.')
            results.append((path.name, actual, predicted))
        if results:
            correct = sum(actual == predicted for _, actual, predicted in results)
            print(f'Handwritten accuracy: {correct}/{len(results)} ({correct/len(results):.2%})')
        return results

In [3]:
mnist = Mnist(BASE / 'sample_weight.pkl')
x_test, y_test = mnist.load_idx(BASE / 't10k-images.idx3-ubyte', BASE / 't10k-labels.idx1-ubyte')
print('Test images:', x_test.shape)
print(f'MNIST test accuracy: {mnist.accuracy(x_test, y_test):.2%}')

Test images: (10000, 784)
MNIST test accuracy: 93.52%


## Optional: prepare your handwritten images
Write a single digit on each image, leaving a clear background. Put the original photographs/drawings in `mnist/raw_digits/` with names such as `2_1.png`. The function below crops, scales to fit a 20 × 20 box, centers on a 28 × 28 grayscale canvas, and saves to `mnist/digits/`. Use tight photographs of the digit with even lighting. Inspect the results. It does not generate handwriting.

In [4]:
def prepare_digit(source, destination):
    with Image.open(source) as original:
        gray = ImageOps.autocontrast(original.convert('L'))
    pixels = np.asarray(gray)
    border = np.concatenate([pixels[0], pixels[-1], pixels[:, 0], pixels[:, -1]])
    if np.median(border) > 127:
        gray = ImageOps.invert(gray)
    mask = gray.point(lambda p: 255 if p > 50 else 0)
    box = mask.getbbox()
    if box is None:
        raise ValueError(f'No digit found in {source}')
    digit = gray.crop(box)
    digit.thumbnail((20, 20), Image.Resampling.LANCZOS)
    canvas = Image.new('L', (28, 28), 0)
    canvas.paste(digit, ((28 - digit.width)//2, (28 - digit.height)//2))
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(destination)

raw_folder = BASE / 'raw_digits'
if raw_folder.exists():
    for source in sorted(raw_folder.glob('*.png')):
        if re.fullmatch(r'[0-9]_[1-5]', source.stem):
            prepare_digit(source, BASE / 'digits' / source.name)
print('Put your own images in mnist/raw_digits/ to prepare them, or use ready-made 28 x 28 grayscale images in mnist/digits/.')

Put your own images in mnist/raw_digits/ to prepare them, or use ready-made 28 x 28 grayscale images in mnist/digits/.


## Test all 50 handwritten images
Rerun this cell after adding your images. Missing images are reported honestly; MNIST dataset images are not substituted.

In [5]:
results = mnist.test_handwritten(BASE / 'digits')

Handwritten test incomplete: 50 of 50 images missing.
Missing: 0_1.png, 0_2.png, 0_3.png, 0_4.png, 0_5.png, 1_1.png, 1_2.png, 1_3.png, 1_4.png, 1_5.png, 2_1.png, 2_2.png, 2_3.png, 2_4.png, 2_5.png, 3_1.png, 3_2.png, 3_3.png, 3_4.png, 3_5.png, 4_1.png, 4_2.png, 4_3.png, 4_4.png, 4_5.png, 5_1.png, 5_2.png, 5_3.png, 5_4.png, 5_5.png, 6_1.png, 6_2.png, 6_3.png, 6_4.png, 6_5.png, 7_1.png, 7_2.png, 7_3.png, 7_4.png, 7_5.png, 8_1.png, 8_2.png, 8_3.png, 8_4.png, 8_5.png, 9_1.png, 9_2.png, 9_3.png, 9_4.png, 9_5.png
